# 🤖 RAG + LLM Integration PoC

## Quick demonstration of complete RAG pipeline:
1. **Load existing vector databases** (OCPP vs All sources)
2. **Retrieve relevant documents** based on user query
3. **Generate answers using LLM** with retrieved context
4. **Compare responses** from different knowledge bases

In [ ]:
# Import required libraries
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("Libraries imported successfully")

/Users/chayan/Developer/chargepoint-emu/aion-poc/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully


In [2]:
# Configuration and setup
config = {
    "ocpp_db": "chrome_db/ocpp_vector_store",
    "all_db": "chrome_db/all_vector_store",
    "embedding_model": OllamaEmbeddings(model="nomic-embed-text"),
    "llm_model": Ollama(model="llama3:8b"),
    "k_results": 3
}

# Load your existing vector databases
def load_vector_stores():
    ocpp_store = Chroma(
        persist_directory=config["ocpp_db"], 
        embedding_function=config["embedding_model"]
    )
    all_store = Chroma(
        persist_directory=config["all_db"], 
        embedding_function=config["embedding_model"]
    )
    return ocpp_store, all_store

ocpp_vectorstore, all_vectorstore = load_vector_stores()
print("Vector stores loaded successfully")

Vector stores loaded successfully


/var/folders/gx/k1m3rp6d757_skvbfw3l_ndc0000gn/T/ipykernel_66246/779809007.py:5: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  "embedding_model": OllamaEmbeddings(model="nomic-embed-text"),
/var/folders/gx/k1m3rp6d757_skvbfw3l_ndc0000gn/T/ipykernel_66246/779809007.py:6: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  "llm_model": Ollama(model="llama3:8b"),
/var/folders/gx/k1m3rp6d757_skvbfw3l_ndc0000gn/T/ipykernel_66246/779809007.p

In [3]:
# Create RAG chains for both vector stores
def create_rag_chain(vectorstore, chain_name):
    """Create a complete RAG chain: Retriever + LLM + Output Parser"""
    
    # Create retriever
    retriever = vectorstore.as_retriever(search_kwargs={"k": config["k_results"]})
    
    # Create prompt template
    prompt = ChatPromptTemplate.from_template("""
    You are a helpful assistant for ChargePoint charging station support.
    Answer the question based ONLY on the provided context.
    
    Context: {context}
    
    Question: {question}
    
    Answer: Provide a clear, specific answer based on the context. If the context doesn't contain enough information, say so.
    """)
    
    # Create RAG chain
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)
    
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | config["llm_model"]
        | StrOutputParser()
    )
    
    print(f"{chain_name} RAG chain created")
    return rag_chain

# Create both RAG chains
ocpp_rag_chain = create_rag_chain(ocpp_vectorstore, "OCPP-only")
all_rag_chain = create_rag_chain(all_vectorstore, "All-sources")

OCPP-only RAG chain created
All-sources RAG chain created


In [4]:
# MINIMAL RAG Comparison (Just Function Definitions - No Execution)
import time
import uuid

def minimal_rag_test(question):
    """Minimal RAG test - one question, two responses, no hanging"""
    
    print(f"\n🔍 QUESTION: {question}")
    print("=" * 60)
    
    session_id = str(uuid.uuid4())[:4]
    print(f"Session: {session_id}")
    
    # OCPP Response
    print(f"\n🔵 OCPP Response:")
    try:
        # Don't create LLM until we actually call the function
        ocpp_docs = ocpp_vectorstore.similarity_search(question, k=2)
        context = "\n\n".join(doc.page_content[:200] for doc in ocpp_docs)  # Limit context size
        
        llm = Ollama(model="llama3:8b")
        prompt = f"Context: {context}\n\nQuestion: {question}\n\nAnswer:"
        response = llm.invoke(prompt)
        print(response)
        del llm  # Immediate cleanup
        
    except Exception as e:
        print(f"Error: {e}")
    
    print(f"\n⏸️ Waiting 2 seconds...")
    time.sleep(2)
    
    # All Sources Response
    print(f"\n🟢 All Sources Response:")
    try:
        all_docs = all_vectorstore.similarity_search(question, k=2)
        context = "\n\n".join(doc.page_content[:200] for doc in all_docs)  # Limit context size
        
        llm = Ollama(model="llama3:8b")
        prompt = f"Context: {context}\n\nQuestion: {question}\n\nAnswer:"
        response = llm.invoke(prompt)
        print(response)
        del llm  # Immediate cleanup
        
    except Exception as e:
        print(f"Error: {e}")
    
    print(f"\n✅ Session {session_id} done")
    print("=" * 60)

# Quick test function
def quick_test():
    """Test with a simple question"""
    minimal_rag_test("What is a ground fault error?")

print("✅ Functions defined (no LLM instances created yet)")
print("💡 Use: quick_test() or minimal_rag_test('your question')")
print("⚡ This should run instantly - no hanging!")

✅ Functions defined (no LLM instances created yet)
💡 Use: quick_test() or minimal_rag_test('your question')
⚡ This should run instantly - no hanging!


In [5]:
# Test a couple questions safely
simple_questions = [
    "What should I do when I see a ground fault error?",
    "Why is charging speed slow?"
]

print("🧪 Testing simple RAG comparison...")
for question in simple_questions:
    minimal_rag_test(question)

🧪 Testing simple RAG comparison...

🔍 QUESTION: What should I do when I see a ground fault error?
Session: b487

🔵 OCPP Response:
According to the OCPP 2.0.1 specification, if you see a ground fault error, you should request logs from the charging station using the `GetLog` message. The charging station is required to provide logs in a standard format (e.g., CSV) to help diagnose and troubleshoot the issue.

⏸️ Waiting 2 seconds...
According to the OCPP 2.0.1 specification, if you see a ground fault error, you should request logs from the charging station using the `GetLog` message. The charging station is required to provide logs in a standard format (e.g., CSV) to help diagnose and troubleshoot the issue.

⏸️ Waiting 2 seconds...

🟢 All Sources Response:

🟢 All Sources Response:
A question from the world of electric vehicle charging!

According to the OCPP (Open Charge Point Protocol) specification, when a ground fault is detected in an electric vehicle charging station, it's conside

## 🎯 Interactive Testing

You can also ask custom questions to both RAG systems:

In [ ]:
# Interactive testing - try your own questions!
def ask_question(question):
    """Quick function to test a single question"""
    compare_rag_responses(question)

# Example usage:
# ask_question("How do I reset a charging station?")
# ask_question("What are the different connector states?")
# ask_question("How to diagnose payment terminal issues?")

print("💡 Use ask_question('your question here') to test individual queries")
print("📊 The function will show responses from both OCPP-only and All-sources RAG systems")